In [0]:

#     Real-Time Analysis of Wikipedia User Navigation Patterns 
#                                                               
#  Team    : Mehak Ali, Tamseel, Afaq Ahmad, Laeeq Ahmad      
#   Supervisor : Muhammad Sadique                               
#   Course  : Big Data Analytics                                
#   Platform: Databricks Free Edition (Serverless)              
#                                                               
#   Architecture: Medallion (Bronze → Silver)                   
#   Data Source : Wikimedia Dumps (2022–2025)                   

print("=" * 65)
print("  Real-Time Analysis of Wikipedia User Navigation Patterns")
print("=" * 65)
print("  Team       : Mehak Ali, Tamseel, Afaq Ahmad, Laeeq Ahmad")
print("  Supervisor : Muhammad Sadique")
print("  Course     : Big Data Analytics")
print("  Platform   : Databricks Free Edition")
print("=" * 65)

  Real-Time Analysis of Wikipedia User Navigation Patterns
  Team       : Mehak Ali, Tamseel, Afaq Ahmad, Laeeq Ahmad
  Supervisor : Muhammad Sadique
  Course     : Big Data Analytics
  Platform   : Databricks Free Edition


In [0]:

# SECTION 0 — GLOBAL CONFIGURATION


CATALOG = "wikipedia_project"
SCHEMA  = "default"
VOLUME  = "wikipedia_raw"

BASE_VOLUME_PATH  = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
CLICKSTREAM_PATH  = f"{BASE_VOLUME_PATH}/clickstream"
PAGEVIEW_PATH     = f"{BASE_VOLUME_PATH}/pageviews"

CLICKSTREAM_BRONZE = f"{CATALOG}.{SCHEMA}.clickstream_bronze"
PAGEVIEW_BRONZE    = f"{CATALOG}.{SCHEMA}.pageviews_bronze"
CLICKSTREAM_SILVER = f"{CATALOG}.{SCHEMA}.clickstream_silver"
PAGEVIEW_SILVER    = f"{CATALOG}.{SCHEMA}.pageviews_silver"

# Sampling strategy
CLICKSTREAM_MONTHS_2022 = ["2022-01", "2022-04", "2022-07", "2022-10"]
CLICKSTREAM_MONTHS_2023 = ["2023-01", "2023-04", "2023-07", "2023-10"]
CLICKSTREAM_MONTHS_2024 = ["2024-01", "2024-04", "2024-07", "2024-10"]
CLICKSTREAM_MONTHS_2025 = ["2025-01", "2025-04", "2025-07", "2025-09"]

PAGEVIEW_FILES_2022 = [
    ("2022-01","01","120000"),("2022-02","01","120000"),("2022-03","01","120000"),
    ("2022-04","01","120000"),("2022-05","01","120000"),("2022-06","01","120000"),
    ("2022-07","01","120000"),("2022-08","01","120000"),("2022-09","01","120000"),
    ("2022-10","01","120000"),("2022-11","01","120000"),("2022-12","01","120000"),
    ("2022-01","15","140000"),("2022-04","15","140000"),("2022-07","15","140000"),
    ("2022-10","15","140000"),("2022-03","08","140000"),("2022-06","08","140000"),
    ("2022-09","08","140000"),("2022-12","08","140000"),
]
PAGEVIEW_FILES_2023 = [
    ("2023-01","01","120000"),("2023-02","01","120000"),("2023-03","01","120000"),
    ("2023-04","01","120000"),("2023-05","01","120000"),("2023-06","01","120000"),
    ("2023-07","01","120000"),("2023-08","01","120000"),("2023-09","01","120000"),
    ("2023-10","01","120000"),("2023-11","01","120000"),("2023-12","01","120000"),
    ("2023-01","15","140000"),("2023-04","15","140000"),("2023-07","15","140000"),
    ("2023-10","15","140000"),("2023-03","08","140000"),("2023-06","08","140000"),
    ("2023-09","08","140000"),("2023-12","08","140000"),
]
PAGEVIEW_FILES_2024 = [
    ("2024-01","01","120000"),("2024-02","01","120000"),("2024-03","01","120000"),
    ("2024-04","01","120000"),("2024-05","01","120000"),("2024-06","01","120000"),
    ("2024-07","01","120000"),("2024-08","01","120000"),("2024-09","01","120000"),
    ("2024-10","01","120000"),("2024-11","01","120000"),("2024-12","01","120000"),
    ("2024-01","15","140000"),("2024-04","15","140000"),("2024-07","15","140000"),
    ("2024-10","15","140000"),("2024-03","08","140000"),("2024-06","08","140000"),
    ("2024-09","08","140000"),("2024-12","08","140000"),
]
PAGEVIEW_FILES_2025 = [
    ("2025-01","01","120000"),("2025-02","01","120000"),("2025-03","01","120000"),
    ("2025-04","01","120000"),("2025-05","01","120000"),("2025-06","01","120000"),
    ("2025-07","01","120000"),("2025-08","01","120000"),("2025-09","01","120000"),
    ("2025-01","15","140000"),("2025-04","15","140000"),("2025-07","15","140000"),
    ("2025-03","08","140000"),("2025-06","08","140000"),("2025-09","08","140000"),
    ("2025-02","15","140000"),("2025-05","15","140000"),("2025-08","15","140000"),
    ("2025-03","20","140000"),("2025-06","20","140000"),
]

print("Global configuration loaded")
print(f" Volume   : {BASE_VOLUME_PATH}")
print(f" Catalog  : {CATALOG}.{SCHEMA}")

Global configuration loaded
 Volume   : /Volumes/wikipedia_project/default/wikipedia_raw
 Catalog  : wikipedia_project.default


In [0]:

# SECTION 1 — INFRASTRUCTURE SETUP
import os, requests
from datetime import datetime

# Create catalog and schema
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
print(f"Catalog  : {CATALOG}")
print(f" Schema   : {SCHEMA}")
print(f" Volume   : {VOLUME}")

# Create directories
for subdir in ["clickstream", "pageviews"]:
    dbutils.fs.mkdirs(f"dbfs:/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{subdir}")
    print(f" Directory: /Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{subdir}")

print("\n Infrastructure ready!")

Catalog  : wikipedia_project
 Schema   : default
 Volume   : wikipedia_raw
 Directory: /Volumes/wikipedia_project/default/wikipedia_raw/clickstream
 Directory: /Volumes/wikipedia_project/default/wikipedia_raw/pageviews

 Infrastructure ready!


In [0]:

# SECTION 2 — DATA DOWNLOAD (2022–2025)
def download_file(url: str, dest_path: str) -> bool:
    """Stream-download url → dest_path. Skips if already exists."""
    if os.path.exists(dest_path):
        size_mb = os.path.getsize(dest_path) / (1024 * 1024)
        print(f"  ⏭  Already exists ({size_mb:.1f} MB): {os.path.basename(dest_path)}")
        return True
    print(f"  ⬇ Downloading: {os.path.basename(dest_path)}")
    try:
        with requests.get(url, stream=True, timeout=300) as r:
            r.raise_for_status()
            downloaded = 0
            with open(dest_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
            print(f"   Done ({downloaded/(1024*1024):.1f} MB): {os.path.basename(dest_path)}")
            return True
    except Exception as e:
        print(f"   Failed: {os.path.basename(dest_path)} — {e}")
        if os.path.exists(dest_path): os.remove(dest_path)
        return False

print(" Download helper ready.")

 Download helper ready.


In [0]:
print("=" * 65)
print(" DOWNLOADING CLICKSTREAM DATA — 2022 to 2025")
print("   Strategy: Quarterly sampling (Jan, Apr, Jul, Oct)")
print("=" * 65)

all_months = (
    CLICKSTREAM_MONTHS_2022 +
    CLICKSTREAM_MONTHS_2023 +
    CLICKSTREAM_MONTHS_2024 +
    CLICKSTREAM_MONTHS_2025
)

cs_results = {}
for month in all_months:
    filename = f"clickstream-enwiki-{month}.tsv.gz"
    url      = f"https://dumps.wikimedia.org/other/clickstream/{month}/{filename}"
    dest     = f"{BASE_VOLUME_PATH}/clickstream/{filename}"
    print(f"\n {month}")
    cs_results[month] = "" if download_file(url, dest) else ""

print("\n--- CLICKSTREAM DOWNLOAD SUMMARY ---")
for year in ["2022", "2023", "2024", "2025"]:
    yr = {m: s for m, s in cs_results.items() if m.startswith(year)}
    print(f"  {year}: {' | '.join([f'{s} {m}' for m, s in yr.items()])}")
print(f"\nTotal: {sum(1 for v in cs_results.values() if v == '')}/{len(all_months)} files")

 DOWNLOADING CLICKSTREAM DATA — 2022 to 2025
   Strategy: Quarterly sampling (Jan, Apr, Jul, Oct)

 2022-01
  ⏭  Already exists (387.3 MB): clickstream-enwiki-2022-01.tsv.gz

 2022-04
  ⏭  Already exists (376.1 MB): clickstream-enwiki-2022-04.tsv.gz

 2022-07
  ⏭  Already exists (384.5 MB): clickstream-enwiki-2022-07.tsv.gz

 2022-10
  ⏭  Already exists (395.9 MB): clickstream-enwiki-2022-10.tsv.gz

 2023-01
  ⏭  Already exists (406.4 MB): clickstream-enwiki-2023-01.tsv.gz

 2023-04
  ⏭  Already exists (400.8 MB): clickstream-enwiki-2023-04.tsv.gz

 2023-07
  ⏭  Already exists (419.4 MB): clickstream-enwiki-2023-07.tsv.gz

 2023-10
  ⏭  Already exists (407.7 MB): clickstream-enwiki-2023-10.tsv.gz

 2024-01
  ⏭  Already exists (432.1 MB): clickstream-enwiki-2024-01.tsv.gz

 2024-04
  ⏭  Already exists (480.7 MB): clickstream-enwiki-2024-04.tsv.gz

 2024-07
  ⏭  Already exists (477.6 MB): clickstream-enwiki-2024-07.tsv.gz

 2024-10
  ⏭  Already exists (477.8 MB): clickstream-enwiki-2024-

In [0]:
print("=" * 65)
print("DOWNLOADING PAGEVIEW DATA — 2022 to 2025")
print("   Strategy: 20 files/year (1st + 15th + 8th sampling)")
print("=" * 65)

all_pv_files = (
    PAGEVIEW_FILES_2022 +
    PAGEVIEW_FILES_2023 +
    PAGEVIEW_FILES_2024 +
    PAGEVIEW_FILES_2025
)

pv_results = {}
for (month, day, hour) in all_pv_files:
    year_part = month.split("-")[0]
    mm        = month.split("-")[1]
    filename  = f"pageviews-{year_part}{mm}{day}-{hour}.gz"
    url       = f"https://dumps.wikimedia.org/other/pageviews/{year_part}/{month}/{filename}"
    dest      = f"{BASE_VOLUME_PATH}/pageviews/{filename}"
    label     = f"{month}-{day}"
    print(f"\n {label}")
    pv_results[label] = "" if download_file(url, dest) else ""

print("\n--- PAGEVIEW DOWNLOAD SUMMARY ---")
for year in ["2022", "2023", "2024", "2025"]:
    count = sum(1 for k, v in pv_results.items() if k.startswith(year) and v == "")
    print(f"  {year}: {count}/20 files downloaded")
print(f"\nTotal: {sum(1 for v in pv_results.values() if v == '')}/{len(all_pv_files)} files")

DOWNLOADING PAGEVIEW DATA — 2022 to 2025
   Strategy: 20 files/year (1st + 15th + 8th sampling)

 2022-01-01
  ⏭  Already exists (46.5 MB): pageviews-20220101-120000.gz

 2022-02-01
  ⏭  Already exists (55.8 MB): pageviews-20220201-120000.gz

 2022-03-01
  ⏭  Already exists (53.0 MB): pageviews-20220301-120000.gz

 2022-04-01
  ⏭  Already exists (53.1 MB): pageviews-20220401-120000.gz

 2022-05-01
  ⏭  Already exists (51.0 MB): pageviews-20220501-120000.gz

 2022-06-01
  ⏭  Already exists (53.4 MB): pageviews-20220601-120000.gz

 2022-07-01
  ⏭  Already exists (50.3 MB): pageviews-20220701-120000.gz

 2022-08-01
  ⏭  Already exists (52.9 MB): pageviews-20220801-120000.gz

 2022-09-01
  ⏭  Already exists (52.7 MB): pageviews-20220901-120000.gz

 2022-10-01
  ⏭  Already exists (50.3 MB): pageviews-20221001-120000.gz

 2022-11-01
  ⏭  Already exists (57.8 MB): pageviews-20221101-120000.gz

 2022-12-01
  ⏭  Already exists (56.3 MB): pageviews-20221201-120000.gz

 2022-01-15
  ⏭  Already ex

In [0]:

# SECTION 3 — BRONZE LAYER (Raw Delta Tables)
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
spark.conf.set("spark.sql.shuffle.partitions", "8")

INGESTION_TIME = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print(" Bronze layer setup complete")
print(f"Ingestion timestamp: {INGESTION_TIME}")

 Bronze layer setup complete
Ingestion timestamp: 2026-05-19 18:35:15


In [0]:
print("=" * 65)
print(" BRONZE — INGESTING CLICKSTREAM")
print("=" * 65)

clickstream_schema = StructType([
    StructField("prev_title",  StringType(), True),
    StructField("curr_title",  StringType(), True),
    StructField("ref_type",    StringType(), True),
    StructField("click_count", LongType(),   True),
])

df_cs = spark.read.csv(
    f"dbfs:{CLICKSTREAM_PATH}/*.tsv.gz",
    sep="\t", header=False, schema=clickstream_schema
) \
.withColumn("source_file",         F.element_at(F.split(F.col("_metadata.file_path"), "/"), -1)) \
.withColumn("year",                F.regexp_extract(F.col("_metadata.file_path"), r"(\d{4})-(\d{2})", 1).cast("int")) \
.withColumn("month",               F.regexp_extract(F.col("_metadata.file_path"), r"(\d{4})-(\d{2})", 2).cast("int")) \
.withColumn("ingestion_timestamp", F.lit(INGESTION_TIME)) \
.withColumn("layer",               F.lit("bronze")) \
.dropna(subset=["prev_title", "curr_title"])

df_cs.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("year", "month") \
    .saveAsTable(CLICKSTREAM_BRONZE)

count = spark.table(CLICKSTREAM_BRONZE).count()
print(f" Table   : {CLICKSTREAM_BRONZE}")
print(f"Rows    : {count:,}")
spark.table(CLICKSTREAM_BRONZE).show(3, truncate=50)

 BRONZE — INGESTING CLICKSTREAM
 Table   : wikipedia_project.default.clickstream_bronze
Rows    : 540,787,215
+------------+---------------+--------+-----------+---------------------------------+----+-----+-------------------+------+
|  prev_title|     curr_title|ref_type|click_count|                      source_file|year|month|ingestion_timestamp| layer|
+------------+---------------+--------+-----------+---------------------------------+----+-----+-------------------+------+
|other-search|     Dighalgram|external|         30|clickstream-enwiki-2024-01.tsv.gz|2024|    1|2026-05-17 19:25:07|bronze|
| other-empty|Maureen_Heppell|external|         19|clickstream-enwiki-2024-01.tsv.gz|2024|    1|2026-05-17 19:25:07|bronze|
|other-search|Black_Rock_mine|external|         43|clickstream-enwiki-2024-01.tsv.gz|2024|    1|2026-05-17 19:25:07|bronze|
+------------+---------------+--------+-----------+---------------------------------+----+-----+-------------------+------+
only showing top 3 row

In [0]:
# Ensure imports are available
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType

print("=" * 65)
print(" BRONZE — INGESTING PAGEVIEWS")
print("=" * 65)

pageview_schema = StructType([
    StructField("project",       StringType(), True),
    StructField("article",       StringType(), True),
    StructField("view_count",    LongType(),   True),
    StructField("response_size", LongType(),   True),
])

df_pv = spark.read.csv(
    f"dbfs:{PAGEVIEW_PATH}/*.gz",
    sep=" ", header=False, schema=pageview_schema
) \
.withColumn("source_file",         F.element_at(F.split(F.col("_metadata.file_path"), "/"), -1)) \
.withColumn("year",                F.regexp_extract(F.col("_metadata.file_path"), r"pageviews-(\d{4})(\d{2})(\d{2})", 1).cast("int")) \
.withColumn("month",               F.regexp_extract(F.col("_metadata.file_path"), r"pageviews-(\d{4})(\d{2})(\d{2})", 2).cast("int")) \
.withColumn("day",                 F.regexp_extract(F.col("_metadata.file_path"), r"pageviews-(\d{4})(\d{2})(\d{2})", 3).cast("int")) \
.withColumn("ingestion_timestamp", F.lit(INGESTION_TIME)) \
.withColumn("layer",               F.lit("bronze")) \
.filter(F.col("project") == "en") \
.dropna(subset=["article", "view_count"])

df_pv.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("year", "month") \
    .saveAsTable(PAGEVIEW_BRONZE)

count = spark.table(PAGEVIEW_BRONZE).count()
print(f"Table   : {PAGEVIEW_BRONZE}")
print(f"Rows    : {count:,}")
spark.table(PAGEVIEW_BRONZE).show(3, truncate=50)

 BRONZE — INGESTING PAGEVIEWS
Table   : wikipedia_project.default.pageviews_bronze
Rows    : 80,180,409
+-------+---------+----------+-------------+----------------------------+----+-----+---+-------------------+------+
|project|  article|view_count|response_size|                 source_file|year|month|day|ingestion_timestamp| layer|
+-------+---------+----------+-------------+----------------------------+----+-----+---+-------------------+------+
|     en|      !!!|         1|            0|pageviews-20230801-120000.gz|2023|    8|  1|2026-05-19 18:35:15|bronze|
|     en|       !?|         1|            0|pageviews-20230801-120000.gz|2023|    8|  1|2026-05-19 18:35:15|bronze|
|     en|!K7_Music|         1|            0|pageviews-20230801-120000.gz|2023|    8|  1|2026-05-19 18:35:15|bronze|
+-------+---------+----------+-------------+----------------------------+----+-----+---+-------------------+------+
only showing top 3 rows


In [0]:

# SECTION 4 — SILVER LAYER (Cleaned, Power BI Ready)
from datetime import datetime

SILVER_TIME = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print(" Silver layer setup complete")
print(f" Silver timestamp: {SILVER_TIME}")
print("   All rows kept — optimized for Power BI DirectQuery")

 Silver layer setup complete
 Silver timestamp: 2026-05-19 18:35:31
   All rows kept — optimized for Power BI DirectQuery


In [0]:
# Ensure imports are available
from pyspark.sql import functions as F

print("=" * 65)
print(" SILVER — CLEANING CLICKSTREAM")
print("=" * 65)

df_cs_silver = spark.table(CLICKSTREAM_BRONZE) \
    .withColumnRenamed("prev_title",  "source_article") \
    .withColumnRenamed("curr_title",  "destination_article") \
    .withColumnRenamed("ref_type",    "referral_type") \
    .withColumnRenamed("click_count", "total_clicks") \
    .withColumn("date", F.to_date(
        F.concat_ws("-",
            F.col("year").cast("string"),
            F.lpad(F.col("month").cast("string"), 2, "0"),
            F.lit("01")), "yyyy-MM-dd")) \
    .withColumn("referral_type",
        F.when(F.col("referral_type") == "link",     "Internal Link")
         .when(F.col("referral_type") == "external", "External Search")
         .when(F.col("referral_type") == "other",    "Other")
         .otherwise("Unknown")) \
    .withColumn("source_article",      F.regexp_replace(F.col("source_article"), "_", " ")) \
    .withColumn("destination_article", F.regexp_replace(F.col("destination_article"), "_", " ")) \
    .withColumn("silver_timestamp",    F.lit(SILVER_TIME)) \
    .withColumn("layer",               F.lit("silver")) \
    .drop("ingestion_timestamp") \
    .select("date","year","month","source_article","destination_article",
            "referral_type","total_clicks","source_file","silver_timestamp","layer") \
    .filter(F.col("total_clicks") > 0) \
    .filter(F.col("source_article").isNotNull()) \
    .filter(F.col("destination_article").isNotNull())

df_cs_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("year", "month") \
    .saveAsTable(CLICKSTREAM_SILVER)

count = spark.table(CLICKSTREAM_SILVER).count()
print(f" Table   : {CLICKSTREAM_SILVER}")
print(f" Rows    : {count:,}")
spark.table(CLICKSTREAM_SILVER).show(3, truncate=50)

In [0]:
# Ensure imports are available
from pyspark.sql import functions as F

print("=" * 65)
print(" SILVER — CLEANING PAGEVIEWS")
print("=" * 65)

df_pv_silver = spark.table(PAGEVIEW_BRONZE) \
    .withColumnRenamed("article",       "article_title") \
    .withColumnRenamed("view_count",    "total_views") \
    .withColumnRenamed("response_size", "response_size_bytes") \
    .withColumn("date", F.to_date(
        F.concat_ws("-",
            F.col("year").cast("string"),
            F.lpad(F.col("month").cast("string"), 2, "0"),
            F.lpad(F.col("day").cast("string"),   2, "0")), "yyyy-MM-dd")) \
    .withColumn("article_title",    F.regexp_replace(F.col("article_title"), "_", " ")) \
    .withColumn("response_size_kb", F.round(F.col("response_size_bytes") / 1024, 2)) \
    .withColumn("silver_timestamp", F.lit(SILVER_TIME)) \
    .withColumn("layer",            F.lit("silver")) \
    .drop("project", "ingestion_timestamp") \
    .select("date","year","month","day","article_title","total_views",
            "response_size_bytes","response_size_kb","source_file",
            "silver_timestamp","layer") \
    .filter(F.col("total_views") > 0) \
    .filter(F.col("article_title").isNotNull())

df_pv_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("year", "month") \
    .saveAsTable(PAGEVIEW_SILVER)

count = spark.table(PAGEVIEW_SILVER).count()
print(f" Table   : {PAGEVIEW_SILVER}")
print(f" Rows    : {count:,}")
spark.table(PAGEVIEW_SILVER).show(3, truncate=50)

In [0]:

# SECTION 5 — FINAL PROJECT SUMMARY

from pathlib import Path

print("\n" + "=" * 65)
print("  Real-Time Analysis of Wikipedia User Navigation Patterns")
print("  COMPLETE PIPELINE SUMMARY")
print("=" * 65)

# Volume files
print("\n RAW DATA (Volume)")
for subdir in ["clickstream", "pageviews"]:
    files = list(Path(f"{BASE_VOLUME_PATH}/{subdir}").glob("*"))
    size  = sum(f.stat().st_size for f in files) / (1024*1024*1024)
    print(f"   {subdir:15} : {len(files):3} files  ({size:.2f} GB)")

# Tables
print("\n DELTA TABLES")
for table, name in [
    (CLICKSTREAM_BRONZE, "Clickstream Bronze"),
    (PAGEVIEW_BRONZE,    "Pageviews Bronze"),
    (CLICKSTREAM_SILVER, "Clickstream Silver"),
    (PAGEVIEW_SILVER,    "Pageviews Silver")
]:
    df    = spark.table(table)
    count = df.count()
    years = sorted([r[0] for r in df.select("year").distinct().collect()])
    size  = spark.sql(f"DESCRIBE DETAIL {table}").select("sizeInBytes").collect()[0][0]
    print(f"   {name:25} : {count:>15,} rows | {size/(1024*1024):>8.1f} MB | {years}")

print("""

              MEDALLION ARCHITECTURE — COMPLETE               

 Data Download  — 96 files (2022–2025)                    
 Bronze Layer   — Raw Delta tables                        
 Silver Layer   — Cleaned, Power BI ready                 

  Power BI Connection:                                        
  Server : dbc-9d89a790-ef48.cloud.databricks.com            
  Path   : /sql/1.0/warehouses/650cfb9a2d38c9ec              
  Tables : clickstream_silver | pageviews_silver              

""")


  Real-Time Analysis of Wikipedia User Navigation Patterns
  COMPLETE PIPELINE SUMMARY

 RAW DATA (Volume)
   clickstream     :  16 files  (6.80 GB)
   pageviews       :  80 files  (4.59 GB)

📊 DELTA TABLES
   Clickstream Bronze        :     540,787,215 rows |   6859.3 MB | [2022, 2023, 2024, 2025]
   Pageviews Bronze          :      80,180,409 rows |    700.6 MB | [2022, 2023, 2024, 2025]
   Clickstream Silver        :     540,787,190 rows |   6855.1 MB | [2022, 2023, 2024, 2025]
   Pageviews Silver          :      80,180,409 rows |    700.1 MB | [2022, 2023, 2024, 2025]


              MEDALLION ARCHITECTURE — COMPLETE               

 Data Download  — 96 files (2022–2025)                    
 Bronze Layer   — Raw Delta tables                        
 Silver Layer   — Cleaned, Power BI ready                 

  Power BI Connection:                                        
  Server : dbc-9d89a790-ef48.cloud.databricks.com            
  Path   : /sql/1.0/warehouses/650cfb9a2d38c9ec    

In [0]:

# SECTION 6 — GOLD LAYER: Pre-Aggregated Dashboard Summary Table
# Purpose: Create a highly optimized, pre-aggregated fact table that combines
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DateType
from datetime import datetime

print("=" * 70)
print(" GOLD LAYER — CREATING dashboard_summary_gold")
print("=" * 70)

GOLD_TABLE = f"{CATALOG}.{SCHEMA}.dashboard_summary_gold"
GOLD_TIME = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# STEP 1: Extract Hour from Pageviews and Aggregate

print("\n[1/4] Processing Pageviews Silver → Extract Hour + Aggregate...")

# Pageview files have format: pageviews-YYYYMMDD-HHMMSS.gz
# Extract hour from filename (HHMMSS format → extract HH)
df_pageviews_hourly = spark.table(PAGEVIEW_SILVER) \
    .withColumn("hour", 
        F.regexp_extract(F.col("source_file"), r"-(\d{2})\d{4}\.gz$", 1).cast("int")) \
    .withColumn("day_of_week", F.date_format(F.col("date"), "EEE")) \
    .groupBy("year", "month", "day", "date", "hour", "day_of_week", "article_title") \
    .agg(
        F.sum("total_views").alias("total_views"),
        F.count("article_title").alias("view_records")
    ) \
    .withColumn("metric_type", F.lit("pageview")) \
    .withColumn("source_article", F.lit(None).cast("string")) \
    .withColumn("referral_type", F.lit(None).cast("string")) \
    .withColumn("total_clicks", F.lit(0).cast("long")) \
    .select(
        "year", "month", "day", "date", "hour", "day_of_week",
        "article_title", "source_article", "referral_type", "metric_type",
        "total_views", "total_clicks", "view_records"
    )

print(f"   Pageviews aggregated at HOURLY grain: {df_pageviews_hourly.count():,} records")

# STEP 2: Aggregate Clickstream at Monthly Grain
print("\n[2/4] Processing Clickstream Silver → Monthly Aggregation...")

# Clickstream data is inherently monthly (no hour/day in source)
# Aggregate at: year, month, article (destination), source, referral_type
df_clickstream_monthly = spark.table(CLICKSTREAM_SILVER) \
    .withColumn("day", F.lit(1).cast("int")) \
    .withColumn("hour", F.lit(None).cast("int")) \
    .withColumn("day_of_week", F.lit(None).cast("string")) \
    .groupBy(
        "year", "month", "day", "date",
        "destination_article", "source_article", "referral_type"
    ) \
    .agg(
        F.sum("total_clicks").alias("total_clicks")
    ) \
    .withColumn("metric_type", F.lit("clickstream")) \
    .withColumn("total_views", F.lit(0).cast("long")) \
    .withColumn("view_records", F.lit(0).cast("long")) \
    .withColumn("hour", F.lit(None).cast("int")) \
    .withColumn("day_of_week", F.lit(None).cast("string")) \
    .withColumnRenamed("destination_article", "article_title") \
    .select(
        "year", "month", "day", "date", "hour", "day_of_week",
        "article_title", "source_article", "referral_type", "metric_type",
        "total_views", "total_clicks", "view_records"
    )

print(f"   Clickstream aggregated at MONTHLY grain: {df_clickstream_monthly.count():,} records")


# STEP 3: Union Both DataFrames into Unified Gold Table

print("\n[3/4] Combining Pageviews + Clickstream → Unified Gold Table...")

df_gold = df_pageviews_hourly.unionByName(df_clickstream_monthly) \
    .withColumn("gold_timestamp", F.lit(GOLD_TIME)) \
    .withColumn("layer", F.lit("gold"))

total_records = df_gold.count()
print(f"   Combined Gold records: {total_records:,}")

# STEP 4: Write to Delta with Modern Optimization (CLUSTER BY AUTO)
print("\n[4/4] Writing to Delta Lake with Auto Liquid Clustering...")

# Write with partitioning + Auto Liquid Clustering (modern alternative to Z-ORDER)
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("year", "month") \
    .saveAsTable(GOLD_TABLE)

# Enable Auto Liquid Clustering for optimal query performance
spark.sql(f"ALTER TABLE {GOLD_TABLE} CLUSTER BY AUTO")
print(f"   ✓ Auto Liquid Clustering enabled on {GOLD_TABLE}")

# Enable Predictive Optimization for automatic maintenance
spark.sql(f"""
    ALTER TABLE {GOLD_TABLE} 
    SET TBLPROPERTIES ('enable_predictive_optimization' = 'true')
""")
print(f"   ✓ Predictive Optimization enabled (auto OPTIMIZE + VACUUM)")

# Run initial OPTIMIZE to apply clustering
spark.sql(f"OPTIMIZE {GOLD_TABLE}")
print(f"   ✓ Initial OPTIMIZE complete")

print("\n" + "=" * 70)
print(" GOLD TABLE CREATION COMPLETE")
print("=" * 70)

# Final Statistics
final_count = spark.table(GOLD_TABLE).count()
size_bytes = spark.sql(f"DESCRIBE DETAIL {GOLD_TABLE}").select("sizeInBytes").first()[0]
size_mb = size_bytes / (1024 * 1024)

print(f"\n Table Name      : {GOLD_TABLE}")
print(f" Total Records   : {final_count:,}")
print(f" Table Size      : {size_mb:.2f} MB")
print(f" Partitioned By  : year, month")
print(f" Clustering      : Auto Liquid (Databricks-managed)")
print(f" Optimization    : Predictive (auto-maintenance)")

# Show sample data
print("\n Sample Data (Top 5 rows):")
spark.table(GOLD_TABLE).orderBy(F.desc("total_views")).show(5, truncate=False)

print("\n Power BI Connection Ready:")
print(f" ✓ Use DirectQuery mode on: {GOLD_TABLE}")
print(f" ✓ All aggregations pre-calculated — no runtime grouping needed")
print(f" ✓ Hourly grain for pageviews, monthly grain for clickstream")

 GOLD LAYER — CREATING dashboard_summary_gold


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6019376788801217>, line 12
      9 print(" GOLD LAYER — CREATING dashboard_summary_gold")
     10 print("=" * 70)
---> 12 GOLD_TABLE = f"{CATALOG}.{SCHEMA}.dashboard_summary_gold"
     13 GOLD_TIME = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
     15 # STEP 1: Extract Hour from Pageviews and Aggregate

NameError: name 'CATALOG' is not defined